# 🌟 Project Overview – Gold Price Prediction Using GRU & LSTM

Forecasting gold prices is a critical task in financial analytics, risk management, and investment strategy. Gold—often viewed as a global economic indicator—exhibits complex temporal behavior, making traditional statistical models insufficient for capturing its dynamic patterns.

In this project, we leverage the power of **Gated Recurrent Units (GRU)** and **Long Short-Term Memory (LSTM)** architectures to model and predict future gold prices based on historical market data.

---

## 🎯 Objectives
- Develop **single-feature** and **multi-feature** deep learning models for gold price forecasting  
- Demonstrate how **GRU and LSTM** models learn **long-range dependencies** in time-series data  
- Analyze and compare model performance across different recurrent architectures  
- Evaluate predictions using **Mean Absolute Error (MAE)** for realistic accuracy measurement  

---

## 🧠 Why Deep Learning for Time-Series Forecasting?
Gold prices are influenced by many unpredictable factors, such as:
- Global economic conditions  
- Supply-demand fluctuations  
- Currency strength  
- Market sentiment  
- Geopolitical events  

These factors create **non-linear, noisy, and highly dynamic patterns**.  
Traditional models (such as ARIMA) struggle to capture such complexity.

GRU and LSTM networks excel because they:
- Retain information across long sequences  
- Capture temporal and sequential dependencies  
- Handle non-linear and noisy data effectively  
- Adapt to both short-term fluctuations and long-term trends  

This makes gated recurrent architectures particularly well-suited for financial time-series modeling.

---

## 🔧 What This Project Includes

### **1️⃣ Single-Feature GRU/LSTM Model**
Uses only **historical gold prices** as input, focusing on temporal momentum and price trend learning.

### **2️⃣ Multi-Feature GRU/LSTM Model**
Incorporates additional market-related variables to provide richer contextual information, enabling the model to learn:
- Interactions between multiple features  
- Macroeconomic influences  
- Volatility and structural price patterns  

---

## 🚀 Key Techniques Used
- Data preprocessing and cleaning  
- Feature scaling using `MinMaxScaler`  
- Time-window sequence generation  
- GRU and LSTM layer stacking  
- Dropout regularization to reduce overfitting  
- Train-test data splitting  
- Model evaluation using MAE  
- Inverse transformation for real-price interpretation  

---

## 📈 End Goal
To build a robust, deep-learning-based forecasting system capable of:
- Accurately predicting gold price movements  
- Demonstrating the strengths of **GRU and LSTM architectures** in time-series forecasting  
- Serving as a solid foundation for future financial prediction and trading models  

This project combines financial insight with modern deep learning techniques to create an effective and practical predictive tool.


## **📥 Download & Load the Dataset**

In this step, we load the gold price dataset into our environment to begin the analysis.  
The data is imported using **Pandas**, one of the most powerful libraries for handling tabular data.

We perform the following actions:
- Read the CSV file containing historical gold prices  
- Convert it into a Pandas DataFrame for easy manipulation  
- Display the dataset to visually inspect its structure  
- Use `df.info()` to understand column types, data ranges, and any missing values  

This step ensures the dataset is correctly loaded and helps us verify that it is clean and ready for preprocessing and modeling.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
data = pd.read_csv('/content/Gold Price.csv')

In [ ]:
df = pd.DataFrame(data)
df

,Date,Price,Open,High,Low,Volume,Chg%
0,2025-01-06,77149,77309,77542,76545,27160,0.44
1,2025-01-03,76813,77246,78600,76613,60,-0.05
2,2025-01-02,76849,76849,76849,76849,0,0.83
3,2025-01-01,76214,76232,76302,76053,60,-0.02
4,2024-12-31,76232,75680,76970,75572,1920,0.95
...,...,...,...,...,...,...,...
2843,2014-01-06,29119,29300,29395,29051,24380,-0.55
2844,2014-01-04,29279,29279,29279,29279,0,-1.51
2845,2014-01-03,29727,30031,30125,29539,3050,-0.83
2846,2014-01-02,29975,29678,30050,29678,3140,1.47


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2848 entries, 0 to 2847
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Date    2848 non-null   object 
 1   Price   2848 non-null   int64  
 2   Open    2848 non-null   int64  
 3   High    2848 non-null   int64  
 4   Low     2848 non-null   int64  
 5   Volume  2848 non-null   int64  
 6   Chg%    2848 non-null   float64
dtypes: float64(1), int64(5), object(1)
memory usage: 155.9+ KB


## **Single Feature**

### 🎯 Extracting the Target Feature (Gold Price)

In this step, we isolate the **Price** column from the dataset.  
This column represents the historical gold prices that we aim to predict using our deep learning models.

By selecting this feature:
- We prepare the target variable for **single-feature time-series forecasting**  
- We extract the raw price values to later apply scaling and sequence generation  
- We simplify the data structure so the model can focus purely on learning temporal patterns from price history  

This forms the foundation for building our first GRU/LSTM model.


In [ ]:
price_col = df['Price']
price_col.values

array([77149, 76813, 76849, ..., 29727, 29975, 29542])

### 🔄 Feature Scaling with MinMaxScaler

Neural networks perform best when input values are normalized, especially in time-series forecasting.  
In this step, we apply **MinMax scaling** to the gold price values, transforming them into a range between **0 and 1**.

Why scaling is important:
- Prevents large numerical values from dominating the learning process  
- Stabilizes gradient updates  
- Speeds up convergence during training  
- Ensures consistency across different features  

Here, we reshape the price column into a 2D array (as required by scikit-learn) and apply the scaler to generate `price_col_scaled`, which will be used to create sequential input windows for the GRU/LSTM model.



In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
price_col_scaled = scaler.fit_transform(price_col.values.reshape(-1, 1))

### 📌 Preparing Time-Series Data with a Sliding Window

To train an RNN/LSTM model, the raw gold-price values must be converted into
supervised learning samples. This is done using a **sliding window technique**.

#### 🔄 How it works
A fixed window size (`time_step = 10`) is chosen. The model receives:
- A sequence of 10 consecutive past prices as input
- The price immediately following that sequence as the target

This transformation reshapes the time series as follows:

- [price₁ → price₁₀]  →  price₁₁
- [price₂ → price₁₁]  →  price₁₂
- [price₃ → price₁₂]  →  price₁₃
- ...

#### 🎯 Why this is important
This structure allows the neural network to learn:
- Temporal dependencies
- Short-term patterns and fluctuations
- How past movements influence future prices

#### 📐 Resulting dataset structure
After processing:
- `X` becomes a sequence array with shape `(samples, time_steps, features)`
- `y` becomes the corresponding next-step target array

This formatting is exactly what recurrent neural networks expect for
sequence-based prediction tasks.


In [ ]:
X = []
y = []
time_step = 10

for i in range(len(price_col_scaled) - (time_step + 1)) :
  X.append(price_col_scaled[i : i + time_step])
  y.append(price_col_scaled[i + time_step])

X = np.array(X)
y = np.array(y)

In [ ]:
y.shape

(2837, 1)

## **Creating Single Feature Model**

### **🧠 Building the Single-Feature Deep Learning Model**

In this section, we import the core components required to construct our **single-feature GRU/LSTM-based neural network** using TensorFlow and Keras.

The imported modules serve the following purposes:

- **Sequential**: Enables the creation of a linear stack of layers, ideal for building recurrent neural networks in a clear and structured way.
- **GRU (Gated Recurrent Unit)**: A gated recurrent architecture designed to efficiently capture temporal dependencies while using fewer parameters than traditional LSTM.
- **LSTM (Long Short-Term Memory)**: A powerful recurrent layer capable of learning long-term dependencies in sequential data.
- **Dense**: Fully connected layers used to map the learned temporal representations to the final output.
- **Dropout**: A regularization technique that helps prevent overfitting by randomly disabling neurons during training.

These components form the foundation for constructing a robust and expressive time-series forecasting model based on historical gold price data.


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, GRU, Dropout, SimpleRNN

### **🏗️ Single-Feature GRU–LSTM Model Architecture**

In this section, we construct a **deep hybrid recurrent neural network** that combines **GRU** and **LSTM** layers to model the temporal dynamics of gold prices using a single input feature (historical price).

#### **🔹 Model Design Philosophy**

Financial time-series data exhibits:
- long-term trends  
- short-term fluctuations  
- non-linear dependencies  

To capture these patterns effectively, we stack multiple recurrent layers with increasing representational power.

---

#### **🔹 GRU Layers (Feature Extraction & Temporal Encoding)**

The model begins with a stack of **GRU (Gated Recurrent Unit)** layers:

- GRUs efficiently capture short- and mid-term temporal dependencies  
- They use fewer parameters than LSTMs, improving training stability  
- Stacking multiple GRU layers allows the model to learn increasingly abstract temporal representations  

Each GRU layer is configured with:
- **200 hidden units** for high expressive capacity  
- `return_sequences=True` to preserve temporal information for deeper layers  

---

#### **🔹 LSTM Layers (Long-Term Dependency Learning)**

After GRU-based feature extraction, the model transitions into **LSTM layers**:

- LSTMs are highly effective at learning long-range dependencies  
- They maintain long-term memory through gated cell states  
- Stacking multiple LSTM layers allows the model to refine and consolidate learned temporal patterns  

The final LSTM layer outputs a fixed-length representation, summarizing the entire input sequence.

---

#### **🔹 Regularization with Dropout**

Dropout layers are used throughout the network to:
- reduce overfitting  
- improve generalization on unseen data  
- stabilize training on noisy financial sequences  

By randomly disabling a fraction of neurons during training, the model avoids memorizing noise and focuses on robust patterns.

---

#### **🔹 Output Layer**

The network concludes with a **Dense layer** using a linear activation function:

- Maps the learned temporal representation to a single continuous value  
- Outputs the predicted gold price for the next time step  

---

#### **🔹 Summary**

This architecture:
- Combines the efficiency of **GRU** with the long-memory strength of **LSTM**  
- Learns both short-term fluctuations and long-term price trends  
- Is well-suited for complex, noisy financial time-series forecasting  

The stacked recurrent design enables the model to extract deep temporal features and deliver accurate gold price predictions.


In [ ]:
model = Sequential()

model.add(GRU(200, input_shape = [None, 1], activation = 'relu', return_sequences = True))
Dropout(0.2),
model.add(GRU(200, activation = 'relu', return_sequences = True))
Dropout(0.2),
model.add(GRU(200, activation = 'relu', return_sequences = True))
Dropout(0.2),
model.add(GRU(200, activation = 'relu', return_sequences = True))
Dropout(0.2),
model.add(GRU(200, activation = 'relu', return_sequences = True))
Dropout(0.2),
model.add(LSTM(200, activation = 'relu', return_sequences = True))
Dropout(0.2),
model.add(LSTM(200, activation = 'relu', return_sequences = True))
Dropout(0.2),
model.add(LSTM(200, activation = 'relu', return_sequences = True))
Dropout(0.2),
model.add(LSTM(200, activation = 'relu', return_sequences = True))
Dropout(0.2),
model.add(LSTM(200, activation = 'relu', return_sequences=True))
Dropout(0.2),
model.add(LSTM(200, activation = 'relu'))
model.add(Dense(1, activation='linear'))

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_8 (GRU)                     │ (None, None, 200)      │       121,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_9 (GRU)                     │ (None, None, 200)      │       241,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_10 (GRU)                    │ (None, None, 200)      │       241,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_11 (GRU)                    │ (None, None, 200)      │       241,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_12 (GRU)                    │ (None, None, 200)      │       241,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_10 (LSTM)                  │ (None, None, 200)      │       320,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_11 (LSTM)                  │ (None, None, 200)      │       320,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_12 (LSTM)                  │ (None, None, 200)      │       320,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_13 (LSTM)                  │ (None, None, 200)      │       320,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_14 (LSTM)                  │ (None, None, 200)      │       320,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_15 (LSTM)                  │ (None, 200)            │       320,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │           201 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,011,601 (11.49 MB)

 Trainable params: 3,011,601 (11.49 MB)

 Non-trainable params: 0 (0.00 B)

### **⚙️ Training Hyperparameters**

In this section, we define the key hyperparameters that control the training process of the deep learning model.

- **Epochs (`epoch = 200`)**  
  Specifies the number of complete passes through the training dataset.  
  A higher number of epochs allows the model to learn complex temporal patterns, while still requiring proper regularization to avoid overfitting.

- **Batch Size (`batch_size = 32`)**  
  Determines the number of samples processed before the model updates its weights.  
  A batch size of 32 provides a good balance between:
  - stable gradient updates  
  - efficient GPU/CPU utilization  
  - improved generalization for time-series data  

These hyperparameters are chosen to ensure stable convergence and effective learning when modeling noisy financial time-series such as gold prices.


In [ ]:
epoch = 200

batch_size = 32

### **🧩 Model Compilation**

Before training, the model must be compiled by specifying the optimization strategy and the loss function.

- **Optimizer (`Adam`)**  
  Adam (Adaptive Moment Estimation) is a widely used optimization algorithm that combines the advantages of:
  - Momentum-based optimization  
  - Adaptive learning rates  

  It is particularly effective for deep recurrent networks, as it:
  - accelerates convergence  
  - handles noisy gradients efficiently  
  - adapts well to non-stationary time-series data  

- **Loss Function (`Mean Squared Error - MSE`)**  
  MSE is a standard loss function for regression tasks and measures the average squared difference between predicted and actual values.

  It strongly penalizes large prediction errors, making it suitable for financial forecasting where large deviations can be costly.

This compilation setup ensures stable training and enables the model to learn accurate representations of gold price movements.


In [ ]:
model.compile(optimizer = 'adam', loss = 'mse')

## **Train Single Feature Model**

### **📊 Train–Test Data Split**

To evaluate the performance of the model on unseen data, we split the dataset into **training** and **testing** subsets.

- **Training Set (80%)**  
  Used to train the GRU–LSTM model and learn temporal patterns from historical gold price data.

- **Testing Set (20%)**  
  Held out from training and used exclusively to evaluate the model’s generalization ability.

The `random_state` parameter ensures reproducibility, allowing the same data split to be obtained across multiple runs.

This separation is essential for assessing how well the model performs on new, unseen time-series data and for preventing overfitting.


In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state = 42)

### **🏋️ Model Training**

In this step, the GRU–LSTM model is trained using the prepared training data.

During training:
- The model processes the input sequences in batches  
- Model weights are updated iteratively to minimize the loss function  
- Temporal patterns in historical gold prices are learned over multiple epochs  

The training configuration uses:
- The predefined **batch size** for stable gradient updates  
- The specified number of **epochs** to allow the model to converge  

This process enables the network to progressively refine its internal representations and improve its ability to forecast future gold prices.


In [ ]:
model.fit(X_train, y_train, batch_size=batch_size, epochs=epoch)

Epoch 1/200
71/71 ━━━━━━━━━━━━━━━━━━━━ 26s 109ms/step - loss: 0.0937
Epoch 2/200
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.0925
Epoch 3/200
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.0803
Epoch 4/200
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.0693
Epoch 5/200
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.0627
Epoch 6/200
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 9.7558e-04
Epoch 7/200
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 4.7285e-04
Epoch 8/200
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 4.2622e-04
Epoch 9/200
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 3.9733e-04
Epoch 10/200
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 4.4131e-04
Epoch 11/200
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 3.7787e-04
Epoch 12/200
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 5.0866e-04
Epoch 13/200
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 3.8967e-04
Epoch 14/200
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 3.6521e-04
Epoch 15/200
71/71 

### **🔮 Generating Predictions**

After training, we use the GRU–LSTM model to **predict gold prices** on the testing dataset.

- `X_test` contains sequences of historical prices that the model has not seen during training  
- The model outputs predicted values for each input sequence  
- These predictions are still in the **scaled range** (0 to 1) and require inverse scaling to interpret in actual price terms  

This step allows us to assess how well the model generalizes and captures temporal patterns in unseen data.


In [ ]:
preds = model.predict(X_test)

18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step


### **🔄 Inverse Scaling to Original Price Range**

The predictions generated by the model (`preds`) and the true test values (`y_test`) are currently in the **scaled range** (0 to 1), due to the earlier MinMax scaling.

To interpret the results in **real-world gold prices**, we perform **inverse transformation** using the same scaler:

- `preds_real` → predicted gold prices in original units  
- `y_test_real` → actual gold prices in original units  

This step allows us to:
- Compare predicted and actual prices in a meaningful scale  
- Evaluate model performance accurately using metrics like MAE  
- Visualize the predictions against the real gold price trends


In [ ]:
preds_real = scaler.inverse_transform(preds)
y_test_real = scaler.inverse_transform(y_test)

### **📏 Evaluating Model Performance – Mean Absolute Error (MAE)**

To assess the accuracy of our single-feature GRU–LSTM model, we use the **Mean Absolute Error (MAE)** metric.

- **MAE** measures the average absolute difference between the predicted and actual gold prices.  
- It is expressed in the same units as the target variable, making it intuitive for financial applications.  
- A lower MAE indicates better predictive performance and more reliable forecasts.


In [ ]:
from sklearn.metrics import mean_absolute_error
single_feature_mae = mean_absolute_error(y_test_real, preds_real)
print(single_feature_mae)

343.59372592979736


## **Multi Feature**

### **🎯 Defining the Target Variable for Multi-Feature Modeling**

In this step, we again isolate the **Price** column from the dataset, which serves as the **target variable** for the multi-feature prediction model.

Although multiple input features will be used, the objective remains the same:
- Predict the future **gold price** based on historical information  

By explicitly defining the target variable:
- We ensure a clear separation between **input features** and **output labels**  
- We maintain consistency across single-feature and multi-feature experiments  
- We enable fair performance comparison between different modeling approaches  

The extracted price values will later be scaled and aligned with the corresponding multi-feature input sequences.


In [ ]:
price_col = df['Price']
price_col.values

array([77149, 76813, 76849, ..., 29727, 29975, 29542])

### **🧩 Selecting Input Features for the Multi-Feature Model**

In this step, we construct the input feature set for the **multi-feature GRU–LSTM model**.

We remove non-numeric and less relevant columns from the dataset:
- **`Date`**: Time indices are implicitly captured through the sequence order and are not directly used as numerical inputs  
- **`Chg%`**: Percentage change values can introduce noise and redundancy, as price movement information is already encoded in the historical price sequence  

The remaining columns represent numerical market features that provide additional context for predicting gold prices, such as price-related indicators and volume-related information.

By selecting these features:
- The model can learn interactions between multiple market variables  
- Temporal dependencies across different signals can be captured  
- Predictive performance can improve compared to single-feature modeling  

The resulting array (`main_features`) will be scaled and transformed into time-series sequences in the next steps.


In [ ]:
main_features = df.drop(columns=['Chg%', 'Date'], axis=1).values
main_features

array([[77149, 77309, 77542, 76545, 27160],
       [76813, 77246, 78600, 76613,    60],
       [76849, 76849, 76849, 76849,     0],
       ...,
       [29727, 30031, 30125, 29539,  3050],
       [29975, 29678, 30050, 29678,  3140],
       [29542, 29435, 29598, 29340,  2930]])

### **🔄 Scaling Target and Input Features**

Before feeding the data into the GRU–LSTM model, we normalize both the **target variable** and the **input features** to ensure stable and efficient training.

#### **🔹 Scaling the Target Variable (Price)**
The gold price values are scaled to the range **[0, 1]** using `MinMaxScaler`.  
This helps the model learn more effectively by preventing large numerical values from dominating the optimization process.

#### **🔹 Scaling the Input Features**
A separate scaler is applied to the multi-feature input matrix:
- Each feature is scaled independently  
- Preserves relative patterns within each feature  
- Ensures all features contribute equally during training  

Using **two different scalers** is important because:
- The target variable must be inverse-transformed later for evaluation  
- Input features do not need to be converted back to original scale  

This normalization step improves convergence speed, numerical stability, and overall predictive performance.


In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
price_col_scaled = scaler.fit_transform(price_col.values.reshape(-1, 1))

In [ ]:
scaler2 = MinMaxScaler()
main_features_scaled = scaler2.fit_transform(main_features)

### **⏱️ Creating Multi-Feature Time-Series Sequences**

Recurrent neural networks require sequential input data.  
In this step, we transform the scaled multi-feature dataset into **time-series sequences** suitable for the GRU–LSTM model.

#### **🔹 Sequence Construction**
- A **sliding window approach** is used with a time step of **100**  
- Each input sample contains:
  - 100 consecutive time steps  
  - Multiple market-related features per time step  
- The corresponding target value is the gold price at the next time step  

Formally:
- `X_multy` → sequences of shape `(time_step, number_of_features)`  
- `y_multy` → next-step gold price labels  

#### **🔹 Why a Larger Time Window?**
A longer time window allows the model to:
- Capture long-term trends  
- Learn seasonal and cyclical patterns  
- Understand delayed market effects  

This sequence generation step prepares the data for effective multi-feature temporal learning.


In [ ]:
X_multy = []
y_multy = []
time_step = 100

for i in range(len(price_col_scaled) - (time_step + 1)) :
  X_multy.append(main_features_scaled[i : i + time_step])
  y_multy.append(price_col_scaled[i + time_step])

X_multy = np.array(X_multy)
y_multy = np.array(y_multy)

## **Creating Multi Feature Model**

### **🧠 Building the Multi-Feature GRU–LSTM Model**

In this section, we import the essential TensorFlow/Keras components required to construct the **multi-feature GRU–LSTM neural network**.

Each imported module plays a specific role in the model design:

- **Sequential**  
  Provides a simple and structured way to stack layers linearly, which is well-suited for recurrent neural network architectures.

- **GRU (Gated Recurrent Unit)**  
  Efficiently captures temporal dependencies across multiple input features while using fewer parameters than traditional LSTM layers, improving training stability.

- **LSTM (Long Short-Term Memory)**  
  Excels at learning long-term dependencies and retaining historical information, making it ideal for modeling complex financial time-series data.

- **Dense**  
  Fully connected layers used to transform the learned temporal representations into the final numerical prediction.

- **Dropout**  
  A regularization technique that reduces overfitting by randomly deactivating neurons during training, which is especially important when working with deep models and noisy financial data.

Together, these components form the foundation for constructing a powerful multi-feature time-series forecasting model capable of learning complex interactions between multiple market variables.


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout, SimpleRNN, GRU

In [ ]:
X_multy.shape

(2747, 100, 5)

### **🏗️ Multi-Feature GRU–LSTM Model Architecture**

In this section, we construct a **deep hybrid recurrent neural network** designed to model complex temporal relationships across **multiple market features**.

The architecture combines the efficiency of **GRU layers** with the long-memory capabilities of **LSTM layers**, allowing the model to learn both short-term market fluctuations and long-term price dynamics.

---

#### **🔹 Input Configuration**
- Each input sample consists of:
  - **100 time steps**
  - **5 market-related features per time step**
- This structure enables the model to analyze how multiple variables evolve together over time.

---

#### **🔹 GRU Layers — Multi-Feature Temporal Encoding**

The first part of the network consists of a stack of **GRU layers**:

- GRUs efficiently process high-dimensional sequential inputs  
- They reduce computational complexity while preserving temporal sensitivity  
- Stacking multiple GRU layers allows hierarchical learning of temporal feature interactions  

Each GRU layer:
- Contains **250 hidden units** for strong representational power  
- Uses `return_sequences=True` to pass full temporal information to subsequent layers  

---

#### **🔹 LSTM Layers — Long-Term Dependency Modeling**

Following the GRU stack, the model transitions into **LSTM layers**:

- LSTMs are specialized for learning long-term dependencies  
- They maintain internal memory states that capture long-range trends  
- Deep stacking enables progressive refinement of temporal representations  

The final LSTM layer outputs a compact representation summarizing the entire input sequence.

---

#### **🔹 Regularization with Dropout**

Dropout is applied throughout the network to:
- Reduce overfitting on noisy financial data  
- Improve generalization on unseen market conditions  
- Encourage the model to learn robust, distributed representations  

This is particularly important given the depth and capacity of the network.

---

#### **🔹 Output Layer**

The final **Dense layer** with linear activation:
- Produces a single continuous output  
- Represents the predicted gold price at the next time step  

---

#### **🔹 Architectural Summary**

This multi-feature GRU–LSTM architecture:
- Captures interactions between multiple market variables  
- Learns both short-term volatility and long-term trends  
- Is well-suited for complex financial time-series forecasting  

The deep stacked design allows the model to extract rich temporal features and deliver accurate gold price predictions.


In [ ]:
model_multy = Sequential()

model_multy.add(GRU(250, input_shape = [None, 5], activation = 'relu', return_sequences = True))
Dropout(0.2),
model_multy.add(GRU(250, activation = 'relu', return_sequences = True))
Dropout(0.2),
model_multy.add(GRU(250, activation = 'relu', return_sequences = True))
Dropout(0.2),
model_multy.add(GRU(250, activation = 'relu', return_sequences = True))
Dropout(0.2),
model_multy.add(GRU(250, activation = 'relu', return_sequences = True))
Dropout(0.2),
model_multy.add(LSTM(250, activation = 'relu', return_sequences = True))
Dropout(0.2),
model_multy.add(LSTM(250, activation = 'relu', return_sequences = True))
Dropout(0.2),
model_multy.add(LSTM(250, activation = 'relu', return_sequences = True))
Dropout(0.2),
model_multy.add(LSTM(250, activation = 'relu', return_sequences = True))
Dropout(0.2),
model_multy.add(LSTM(250, activation = 'relu', return_sequences=True))
Dropout(0.2),
model_multy.add(LSTM(250, activation = 'relu'))
model_multy.add(Dense(1, activation='linear'))

model_multy.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, None, 250)      │       192,750 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, None, 250)      │       376,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_2 (GRU)                     │ (None, None, 250)      │       376,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_3 (GRU)                     │ (None, None, 250)      │       376,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_4 (GRU)                     │ (None, None, 250)      │       376,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, None, 250)      │       501,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, None, 250)      │       501,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, None, 250)      │       501,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, None, 250)      │       501,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, None, 250)      │       501,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 250)            │       501,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           251 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, None, 250)      │       192,750 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, None, 250)      │       376,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_2 (GRU)                     │ (None, None, 250)      │       376,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_3 (GRU)                     │ (None, None, 250)      │       376,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_4 (GRU)                     │ (None, None, 250)      │       376,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, None, 250)      │       501,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, None, 250)      │       501,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, None, 250)      │       501,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, None, 250)      │       501,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, None, 250)      │       501,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 250)            │       501,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           251 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

...

 Total params: 4,705,001 (17.95 MB)

 Trainable params: 4,705,001 (17.95 MB)

 Trainable params: 4,705,001 (17.95 MB)

 Non-trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

### **⚙️ Training Hyperparameters for the Multi-Feature Model**

In this section, we define the key hyperparameters used to train the **multi-feature GRU–LSTM model**.

- **Epochs (`epoch = 200`)**  
  Specifies the number of complete passes through the training dataset.  
  A higher number of epochs allows the deep recurrent network to fully learn complex temporal patterns and feature interactions present in multi-dimensional financial data.

- **Batch Size (`batch_size = 64`)**  
  Determines how many samples are processed before the model updates its weights.  
  A slightly larger batch size is chosen for the multi-feature model to:
  - stabilize gradient updates  
  - improve computational efficiency  
  - accommodate the higher dimensionality of the input data  

These hyperparameter values provide a balance between learning capacity, training stability, and computational performance for multi-feature time-series forecasting.


In [ ]:
epoch = 200

batch_size = 64

### **🧩 Compiling the Multi-Feature Model**

Before training, the multi-feature GRU–LSTM model must be compiled by defining the optimization strategy and the loss function.

- **Optimizer (`Adam`)**  
  Adam is an adaptive optimization algorithm that efficiently adjusts the learning rate for each parameter.  
  It is particularly well-suited for deep recurrent networks because it:
  - handles noisy gradients effectively  
  - accelerates convergence  
  - adapts well to non-stationary financial time-series data  

- **Loss Function (`Mean Squared Error - MSE`)**  
  MSE is a standard regression loss function that measures the average squared difference between predicted and actual gold prices.  
  It penalizes larger errors more heavily, encouraging the model to learn precise price predictions.

This compilation setup ensures stable training and enables the model to learn complex relationships between multiple market features and gold price movements.


In [ ]:
model_multy.compile(optimizer = 'adam', loss = 'mse')

## **Train Multi Feature Model**

### **📊 Train–Test Split for the Multi-Feature Model**

To evaluate the generalization ability of the multi-feature GRU–LSTM model, the dataset is divided into **training** and **testing** subsets.

- **Training Set (80%)**  
  Used to train the model and learn temporal relationships across multiple market features.

- **Testing Set (20%)**  
  Held out from training and used to assess model performance on unseen data.

The `random_state` parameter ensures reproducibility by maintaining a consistent data split across multiple runs.

This separation is essential for validating the model’s ability to generalize to new market conditions and prevents overly optimistic performance estimates.


In [ ]:
from sklearn.model_selection import train_test_split
X_train_multy, X_test_multy, y_train_multy, y_test_multy = train_test_split(X_multy, y_multy, test_size=0.2, random_state = 42)

### **🏋️ Training the Multi-Feature GRU–LSTM Model**

In this step, the multi-feature GRU–LSTM model is trained using the prepared training dataset while simultaneously monitoring its performance on unseen validation data.

During training:
- The model learns temporal patterns and feature interactions from the training set  
- Weight updates are performed in mini-batches for stable optimization  
- Validation loss is tracked at each epoch to evaluate generalization performance  

#### **🔹 Key Benefits of Using Validation Data**
- Helps detect overfitting early  
- Provides insight into how well the model performs on unseen data  
- Enables informed hyperparameter tuning  

By training the model over multiple epochs, the network progressively refines its internal representations, improving its ability to predict future gold prices using multiple market indicators.


In [ ]:
model_multy.fit(X_train_multy, y_train_multy, batch_size=batch_size, epochs=epoch, validation_data=(X_test_multy, y_test_multy))

Epoch 1/200
35/35 ━━━━━━━━━━━━━━━━━━━━ 41s 589ms/step - loss: 0.0863 - val_loss: 0.0586
Epoch 2/200
35/35 ━━━━━━━━━━━━━━━━━━━━ 8s 230ms/step - loss: 0.0373 - val_loss: 0.0054
Epoch 3/200
35/35 ━━━━━━━━━━━━━━━━━━━━ 8s 230ms/step - loss: 0.0036 - val_loss: 0.0018
Epoch 4/200
35/35 ━━━━━━━━━━━━━━━━━━━━ 8s 232ms/step - loss: 0.0015 - val_loss: 6.3100e-04
Epoch 5/200
35/35 ━━━━━━━━━━━━━━━━━━━━ 8s 239ms/step - loss: 7.8865e-04 - val_loss: 0.0010
Epoch 6/200
35/35 ━━━━━━━━━━━━━━━━━━━━ 8s 232ms/step - loss: 8.6767e-04 - val_loss: 9.3032e-04
Epoch 7/200
35/35 ━━━━━━━━━━━━━━━━━━━━ 8s 233ms/step - loss: 7.9713e-04 - val_loss: 5.8251e-04
Epoch 8/200
35/35 ━━━━━━━━━━━━━━━━━━━━ 8s 240ms/step - loss: 6.8469e-04 - val_loss: 6.1665e-04
Epoch 9/200
35/35 ━━━━━━━━━━━━━━━━━━━━ 8s 232ms/step - loss: 7.4687e-04 - val_loss: 0.0013
Epoch 10/200
35/35 ━━━━━━━━━━━━━━━━━━━━ 8s 237ms/step - loss: 0.0013 - val_loss: 7.3091e-04
Epoch 11/200
35/35 ━━━━━━━━━━━━━━━━━━━━ 9s 243ms/step - loss: 8.0188e-04 - val_loss: 7.0

### **🔮 Generating Predictions with the Multi-Feature Model**

After training, the multi-feature GRU–LSTM model is used to generate predictions on the testing dataset.

- `X_test_multy` contains sequences of multi-dimensional market features that were not seen during training  
- The model outputs predicted values corresponding to future gold prices  
- The predictions are produced in the **scaled space** and must be inverse-transformed for real-world interpretation  

This step allows us to evaluate how effectively the model captures complex interactions between multiple features and generalizes to unseen market data.


In [ ]:
preds_multy = model_multy.predict(X_test_multy)

18/18 ━━━━━━━━━━━━━━━━━━━━ 7s 206ms/step


### **🔄 Inverse Scaling for Multi-Feature Predictions**

The predictions produced by the multi-feature GRU–LSTM model, as well as the corresponding true values, are currently in the **normalized range** due to MinMax scaling.

To interpret the results in terms of actual gold prices, we apply **inverse transformation** using the scaler fitted on the target variable:

- `preds_real_multy` → predicted gold prices in original units  
- `y_test_real_multy` → actual gold prices in original units  

This step is essential for:
- Meaningful performance evaluation  
- Real-world interpretability of the model’s predictions  
- Direct comparison between predicted and observed gold price movements


In [ ]:
preds_real_multy = scaler.inverse_transform(preds_multy)
y_test_real_multy = scaler.inverse_transform(y_test_multy)

### **📏 Evaluating the Multi-Feature Model – Mean Absolute Error (MAE)**

To quantify the performance of the multi-feature GRU–LSTM model, we compute the **Mean Absolute Error (MAE)** between the predicted and actual gold prices.

- **MAE** represents the average absolute difference between model predictions and true values  
- It is expressed in the same units as gold price, making it intuitive and easy to interpret  
- Lower MAE values indicate better predictive accuracy and stronger generalization performance  

The computed MAE serves as a final evaluation metric for the multi-feature model and allows direct comparison with the single-feature model to assess the impact of incorporating additional market information.


In [ ]:
from sklearn.metrics import mean_absolute_error
multy_features_mae = mean_absolute_error(y_test_real_multy, preds_real_multy)
print(multy_features_mae)

154.19883167613645


# **📊 Creating a Table for Visualizing and Comparing Methods**

In [ ]:
import plotly.graph_objects as go

# Calculate evaluation metrics for each model
models = ['Single_Feature', 'Multy_Features']

MAE = [single_feature_mae, multy_features_mae]

# Plotting the comparison using Plotly
fig = go.Figure()

# Add traces for each evaluation metric
fig.add_trace(go.Bar(
    x=models,
    y=MAE,
    name='MAE',
    marker_color='blue'
))

# Customize the layout of the figure
fig.update_layout(
    title="Model Performance Comparison",
    barmode='group',  # Group the bars together for each model
    xaxis_title="Models",
    yaxis_title="Scores",
    template="plotly_dark",  # Dark theme for the plot
    legend_title="Metrics",
    font=dict(family="Arial, sans-serif", size=14),
    height=500,  # Adjust the height of the chart
    width=800,  # Adjust the width of the chart
)

# Show the plot
fig.show()

## 📊 Model Performance Comparison

To clearly compare the predictive performance of the implemented models, we summarize the evaluation results in a structured **Pandas DataFrame**.

### 🔹 What This Step Does
- Stores the **Mean Absolute Error (MAE)** for each model  
- Organizes results into a tabular format for easy comparison  
- Sorts models by MAE in ascending order (lower is better)  

This allows us to objectively assess which modeling approach delivers more accurate gold price predictions.

---

### 📈 Performance Summary

| Model            | MAE        |
|------------------|------------|
| Multi-Feature    | **154.20** |
| Single-Feature   | 343.59     |

---

### 🔍 Interpretation
- The **multi-feature GRU–LSTM model** significantly outperforms the single-feature model  
- Incorporating additional market variables reduces prediction error by more than **50%**  
- This highlights the importance of leveraging multiple correlated signals when forecasting complex financial time-series  

The comparison demonstrates that richer feature representations lead to more accurate and reliable gold price predictions.


In [ ]:
# Create a pandas DataFrame for the evaluation metrics
performance_df = pd.DataFrame({
    'Model': models,
    'MAE': MAE
})

# Sort the table by Accuracy (or any other metric, e.g., 'F1 Score')
performance_df_sorted = performance_df.sort_values(by='MAE', ascending=True)

# Display the sorted table
print(performance_df_sorted.to_string(index=False))

         Model        MAE
Multy_Features 154.198832
Single_Feature 343.593726


## ✅ Conclusion

In this project, we developed and evaluated deep learning models based on **GRU and LSTM architectures** to forecast gold prices using both **single-feature** and **multi-feature** approaches.

The experimental results clearly demonstrate the impact of incorporating additional market information:

| Model           | MAE |
|-----------------|------|
| Multi-Feature   | **154.20** |
| Single-Feature  | 343.59 |

### 🔍 Key Insights
- The **multi-feature GRU–LSTM model** significantly outperforms the single-feature model, achieving a **more than 50% reduction in Mean Absolute Error**.
- Incorporating multiple correlated market features enables the model to capture:
  - richer temporal dependencies  
  - complex feature interactions  
  - long-term price dynamics more effectively  
- The single-feature model, while capable of learning basic price trends, lacks sufficient contextual information to accurately model real-world gold price movements.

### 📈 Implications
These findings highlight the importance of **feature richness** in financial time-series forecasting.  
Gold prices are influenced by multiple interacting factors, and deep recurrent architectures benefit greatly from access to diverse market signals.

### 🚀 Final Remarks
The results confirm that **GRU–LSTM hybrid architectures**, when combined with multi-feature inputs, provide a powerful and effective framework for modeling complex financial time-series data.  
This project establishes a solid foundation for advanced research and practical applications in algorithmic trading, risk management, and financial forecasting.

---

### 🔮 Future Work
- Integrating additional macroeconomic indicators (e.g., USD index, interest rates)  
- Hyperparameter optimization and automated tuning  
- Exploring attention-based or transformer models  
- Extending predictions to multi-step forecasting horizons  
